# The Script: WR Player-Game Data Exploration

> This notebook audits the canonical regular-season WR player-game dataset used by The Script’s receiving-yards forecasting pipeline.

**Prediction horizon:** 24 hours before kickoff  
**Primary target:** `receiving_yards`  
**Training observation:** snap-verified regular-season WR games  
**Historical span:** 2016–2025

This notebook documents the canonical source table only. Leakage-safe features are built separately by `the_script.features.player`; model fitting and validation are covered by the model-comparison workflow and CLI.

In [1]:
from pathlib import Path



import polars as pl



PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TABLE_PATH = PROJECT_ROOT / "data" / "processed" / "wr_player_games.parquet"

player_games = pl.read_parquet(TABLE_PATH)

player_games.shape

(22641, 18)

## Inspect Ten Rows

Confirm player IDs, teams, schedule-derived opponents, dates, home/away status, rest, and receiving outcomes before feature engineering.

In [2]:
player_games.head(10)

game_id,game_date,season,week,player_id,team,opponent,home_game,days_rest,position,receiving_yards,targets,receptions,receiving_touchdowns,target_share,air_yards_share,wopr,snap_share
str,date,i32,i32,str,str,str,bool,i32,str,i32,i32,i32,i32,f64,f64,f64,f64
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0025396""","""CAR""","""DEN""",false,7,"""WR""",5,1,1,0,0.032258,0.02008,0.062443,0.62
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0026739""","""DEN""","""CAR""",true,7,"""WR""",0,1,0,0,0.038462,0.064,0.102492,0.52
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0027685""","""DEN""","""CAR""",true,7,"""WR""",49,8,5,0,0.307692,0.488,0.803138,0.86
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0027874""","""DEN""","""CAR""",true,7,"""WR""",48,6,4,0,0.230769,0.32,0.570154,0.76
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0031118""","""CAR""","""DEN""",false,7,"""WR""",11,2,1,0,0.064516,0.008032,0.102397,0.53
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0031344""","""DEN""","""CAR""",true,7,"""WR""",0,0,0,0,0.0,0.0,0.0,0.29
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0031359""","""CAR""","""DEN""",false,7,"""WR""",91,12,6,1,0.387097,0.429719,0.881448,0.71
"""2016_01_CAR_DEN""",2016-09-08,2016,1,"""00-0032055""","""CAR""","""DEN""",false,7,"""WR""",9,4,1,0,0.129032,0.196787,0.331299,0.53
"""2016_01_BUF_BAL""",2016-09-11,2016,1,"""00-0020337""","""BAL""","""BUF""",true,7,"""WR""",19,8,5,0,0.25,0.164659,0.490261,0.66


In [3]:
canonical_ids = ["game_id", "game_date", "player_id", "team", "opponent"]



quality_summary = player_games.select(

    pl.len().alias("rows"),

    pl.struct(["game_id", "player_id"]).n_unique().alias("unique_player_games"),

    pl.sum_horizontal(pl.col(canonical_ids).null_count()).alias("null_canonical_ids"),

    pl.col("season").min().alias("first_season"),

    pl.col("season").max().alias("last_season"),

    pl.col("receiving_yards").min().alias("min_receiving_yards"),

    pl.col("receiving_yards").max().alias("max_receiving_yards"),

    pl.col("home_game").mean().alias("home_share"),

)



assert quality_summary["rows"][0] == quality_summary["unique_player_games"][0]

assert quality_summary["null_canonical_ids"][0] == 0

assert quality_summary["first_season"][0] == 2016

assert quality_summary["last_season"][0] == 2025

quality_summary

rows,unique_player_games,null_canonical_ids,first_season,last_season,min_receiving_yards,max_receiving_yards,home_share
u32,u32,u32,i32,i32,i32,i32,f64
22641,22641,0,2016,2025,-13,300,0.499801


In [4]:
player_games.group_by("season").len().sort("season")

season,len
i32,u32
2016,2193
2017,2157
2018,2132
2019,2152
2020,2202
2021,2391
2022,2299
2023,2394
2024,2353


## Audit Notes

- The refreshed canonical table contains 22,641 unique, snap-verified WR-game rows from 2016 through 2025.
- Canonical identifiers are complete, and `(game_id, player_id)` is unique.
- Receiving yards may be negative; these are valid outcomes and must not be dropped.
- The table contains game outcomes and pregame role fields. Model features are generated separately and must use shifted, prior-game values only.
- This notebook audits WR data; the application also trains and serves QB, RB, and TE forecasts.